# LeetCode #1397: Find All Good Strings

https://leetcode.com/problems/find-all-good-strings/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(26^n \cdot n)$ | $O(n)$ |
| **Optimal: Digit DP + KMP ★** | $O(n \cdot |evil| \cdot 26)$ | $O(n \cdot |evil|)$ |

---

## Understanding the Methods

### Brute Force
Enumerate every string of length `n`, check it is in `[s1, s2]`, and verify it contains no substring matching `evil`. The search space $26^n$ is completely impractical.

### Optimal: Digit DP + KMP ★
Use digit DP to count strings in `[s1, s2]` that avoid `evil`. State: `(position, KMP state in evil, tight_lo, tight_hi)`. Precompute KMP failure function so we can advance the evil-match state in $O(1)$ per character. Count strings up to `s2` then subtract strings up to `s1` (adjusted by 1).

**Why this is better than Brute Force:** The DP has $O(n \cdot |evil|)$ states, each taking $O(26)$ transitions — polynomial versus exponential.

**Constraints:**
* `s1.length == s2.length == n`
* $1 \leq n \leq 500$
* `evil.length` $\leq 50$
* All strings lowercase English

## Solutions

### C#

In [ ]:
public class Solution {
    const int MOD = 1_000_000_007;

    public int FindGoodStrings(int n, string s1, string s2, string evil) {
        // Precompute KMP failure function for evil
        int[] fail = KmpFail(evil);
        int m = evil.Length;

        // Count strings <= s in [a..z]^n that don't contain evil
        int Count(string s, bool inclusive) {
            // dp[evilState][tightHi] — rolling over positions
            var dp = new long[m, 2];
            dp[0, 1] = 1; // Start at position 0, tight, no evil matched yet

            for (int pos = 0; pos < n; pos++) {
                var ndp = new long[m, 2];
                char limit = s[pos];
                for (int es = 0; es < m; es++) {
                    for (int tight = 0; tight <= 1; tight++) {
                        if (dp[es, tight] == 0) continue;
                        char maxC = tight == 1 ? limit : 'z';
                        for (char c = 'a'; c <= maxC; c++) {
                            // Advance KMP state
                            int nes = es;
                            while (nes > 0 && evil[nes] != c) nes = fail[nes - 1];
                            if (evil[nes] == c) nes++;
                            if (nes == m) continue; // evil substring found — skip
                            int nTight = tight == 1 && c == limit ? 1 : 0;
                            ndp[nes, nTight] = (ndp[nes, nTight] + dp[es, tight]) % MOD;
                        }
                    }
                }
                dp = ndp;
            }
            long res = 0;
            for (int es = 0; es < m; es++) {
                // tight=1 counts the string s itself; include only if inclusive
                res = (res + dp[es, 0] + (inclusive ? dp[es, 1] : 0)) % MOD;
            }
            return (int)res;
        }

        // f(s2) - f(s1) + (s1 itself is good ? 1 : 0)
        int ans = (Count(s2, true) - Count(s1, false) + MOD) % MOD;
        // Check if s1 itself contains evil
        if (!s1.Contains(evil)) ans = (ans + 1) % MOD;
        return ans;
    }

    int[] KmpFail(string p) {
        int[] f = new int[p.Length];
        for (int i = 1; i < p.Length; i++) {
            int j = f[i - 1];
            while (j > 0 && p[j] != p[i]) j = f[j - 1];
            if (p[j] == p[i]) j++;
            f[i] = j;
        }
        return f;
    }
}

### Python

In [ ]:
from functools import lru_cache

class Solution:
    def find_good_strings(self, n: int, s1: str, s2: str, evil: str) -> int:
        MOD = 10**9 + 7
        m = len(evil)

        # Precompute KMP failure function for evil
        fail = [0] * m
        j = 0
        for i in range(1, m):
            while j > 0 and evil[j] != evil[i]:
                j = fail[j - 1]
            if evil[j] == evil[i]:
                j += 1
            fail[i] = j

        def kmp_next(es: int, c: str) -> int:
            # Advance KMP state from es on character c
            while es > 0 and evil[es] != c:
                es = fail[es - 1]
            return es + 1 if evil[es] == c else 0

        def count(s: str, inclusive: bool) -> int:
            @lru_cache(maxsize=None)
            def dp(pos, es, tight):
                if es == m:
                    return 0  # evil found — invalid
                if pos == n:
                    return 1  # complete valid string
                limit = s[pos] if tight else 'z'
                total = 0
                for c in map(chr, range(ord('a'), ord(limit) + 1)):
                    nes = kmp_next(es, c)
                    if nes == m:
                        continue
                    ntight = tight and (c == limit)
                    total += dp(pos + 1, nes, ntight)
                return total % MOD

            res = dp(0, 0, True)
            if not inclusive:
                # Subtract the string s itself if it appears in the count
                pass
            dp.cache_clear()
            return res % MOD

        # Count [s1..s2]: f(s2, inclusive) - f(s1, inclusive=False) + (s1 good ? 1 : 0)
        ans = (count(s2, True) - count(s1, True) + MOD) % MOD
        # Re-add s1 if it's a good string
        def is_good(s):
            es = 0
            for c in s:
                es = kmp_next(es, c)
                if es == m:
                    return False
            return True

        if is_good(s1):
            ans = (ans + 1) % MOD
        return ans

### Go

In [ ]:
func findGoodStrings(n int, s1 string, s2 string, evil string) int {
    const MOD = 1_000_000_007
    m := len(evil)

    // Precompute KMP failure function for evil
    fail := make([]int, m)
    for i, j := 1, 0; i < m; i++ {
        for j > 0 && evil[j] != evil[i] { j = fail[j-1] }
        if evil[j] == evil[i] { j++ }
        fail[i] = j
    }

    kmpNext := func(es int, c byte) int {
        for es > 0 && evil[es] != c { es = fail[es-1] }
        if evil[es] == c { es++ }
        return es
    }

    // dp[pos][evilState][tight] — memoised digit DP
    type Key struct{ pos, es, tight int }
    memo := map[Key]int{}
    var dp func(pos, es, tight int, s string) int
    dp = func(pos, es, tight int, s string) int {
        if es == m { return 0 } // evil found
        if pos == n { return 1 } // valid complete string
        key := Key{pos, es, tight}
        if v, ok := memo[key]; ok { return v }
        limit := byte('z')
        if tight == 1 { limit = s[pos] }
        total := 0
        for c := byte('a'); c <= limit; c++ {
            nes := kmpNext(es, c)
            if nes == m { continue }
            nt := 0
            if tight == 1 && c == limit { nt = 1 }
            total = (total + dp(pos+1, nes, nt, s)) % MOD
        }
        memo[key] = total
        return total
    }

    count := func(s string) int {
        clear(memo)
        return dp(0, 0, 1, s)
    }

    // Subtract count(s1 exclusive) from count(s2 inclusive)
    isGood := func(s string) bool {
        es := 0
        for i := 0; i < len(s); i++ {
            es = kmpNext(es, s[i])
            if es == m { return false }
        }
        return true
    }

    ans := (count(s2) - count(s1) + MOD) % MOD
    if isGood(s1) { ans = (ans + 1) % MOD }
    return ans
}

### Rust

In [ ]:
use std::collections::HashMap;

impl Solution {
    pub fn find_good_strings(n: i32, s1: String, s2: String, evil: String) -> i32 {
        const MOD: i64 = 1_000_000_007;
        let n = n as usize;
        let evil: Vec<u8> = evil.bytes().collect();
        let m = evil.len();

        // Precompute KMP failure function for evil
        let mut fail = vec![0usize; m];
        let mut j = 0usize;
        for i in 1..m {
            while j > 0 && evil[j] != evil[i] { j = fail[j - 1]; }
            if evil[j] == evil[i] { j += 1; }
            fail[i] = j;
        }

        let kmp_next = |es: usize, c: u8| -> usize {
            let mut es = es;
            while es > 0 && evil[es] != c { es = fail[es - 1]; }
            if evil[es] == c { es += 1; }
            es
        };

        let count = |s: &[u8]| -> i64 {
            let mut memo: HashMap<(usize, usize, bool), i64> = HashMap::new();
            fn dp(pos: usize, es: usize, tight: bool, s: &[u8], n: usize, m: usize,
                  evil: &[u8], fail: &[usize], memo: &mut HashMap<(usize,usize,bool),i64>,
                  kmp_next: &dyn Fn(usize, u8) -> usize) -> i64 {
                if es == m { return 0; }
                if pos == n { return 1; }
                if let Some(&v) = memo.get(&(pos, es, tight)) { return v; }
                let limit = if tight { s[pos] } else { b'z' };
                let mut total = 0i64;
                for c in b'a'..=limit {
                    let nes = kmp_next(es, c);
                    if nes == m { continue; }
                    let nt = tight && c == limit;
                    total = (total + dp(pos+1, nes, nt, s, n, m, evil, fail, memo, kmp_next)) % 1_000_000_007;
                }
                memo.insert((pos, es, tight), total);
                total
            }
            dp(0, 0, true, s, n, m, &evil, &fail, &mut memo, &kmp_next)
        };

        let is_good = |s: &[u8]| -> bool {
            let mut es = 0;
            for &c in s { es = kmp_next(es, c); if es == m { return false; } }
            true
        };

        let s1b: Vec<u8> = s1.bytes().collect();
        let s2b: Vec<u8> = s2.bytes().collect();
        let mut ans = (count(&s2b) - count(&s1b) + MOD) % MOD;
        if is_good(&s1b) { ans = (ans + 1) % MOD; }
        ans as i32
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `n=2, s1="aa", s2="da", evil="b"`
Strings of length 2 in ["aa".."da"] that don't contain 'b'. KMP state for 'b' is length 1; any string with 'b' is excluded. Count all two-char strings in range minus those with 'b'. Answer: **51**.

### 2. Slightly Complex
**Input:** `n=8, s1="leetcode", s2="leetcode", evil="leet"`
Only one candidate: "leetcode" itself. It starts with "leet" = evil. So it contains evil and is not good. Answer: **0**.

### 3. Edge Case: Time Factor
**Input:** `n=500, |evil|=50`
The DP table has $500 \times 50 \times 2 = 50{,}000$ states, each iterating 26 characters with a KMP lookup — $\approx 1.3 \times 10^6$ operations. Maximum input size.

### 4. Edge Case: Space Factor
**Input:** `n=500, |evil|=50`
Memoisation stores $\approx 50{,}000$ entries; the KMP failure array has 50 entries. Total $O(n \cdot |evil|)$ space — the worst case for memory.

### 5. Almost-Impossible but Plausible
**Input:** `n=1, s1="a", s2="z", evil="a"`
Every single-character string from "b" to "z" is good (24 strings). "a" is excluded because it equals evil. Answer: **25**.